# MATH GR 5320 — Portfolio Risk System: Comprehensive Demo

**Columbia University · Spring 2026**

This notebook covers all 15 sections from the course formula sheet, tracing from core market-risk measures through credit and regulatory capital. Every calculation uses the production `src/` modules — the same code the Streamlit app calls — so notebook results and front-end outputs match exactly.

| # | Topic | HW source | Key target |
|---|-------|-----------|------------|
| 1 | Coverage matrix & risk-measure theory | HW03 Q1 | coherence axioms, VaR vs ES |
| 2 | European option pricing & delta | Repo HW5_BS case | price 17.6246, Δ 0.6643 |
| 3 | Delta-hedge intuition | Repo HW3 Intel | 1 873 calls to write |
| 4 | Historical scenario VaR/ES | HW03 scenario | VaR₉₀ 3 931, ES₈₀ 3 429 |
| 5 | Single-stock GBM VaR | HW04 Q1 | 5-day 99% VaR ≈ 19 037 |
| 6 | Two-stock parametric VaR | HW04 Q2 | 2-week 99% VaR ≈ 9 007 |
| 7 | Rolling window vs EWMA calibration | HW05 | λ(2y) ≈ 0.9968 |
| 8 | Historical AAPL/CAT VaR & ES | Bloomberg CSVs | diversification visible |
| 9 | Monte Carlo VaR & ES | MC engine | ES ≥ VaR, ES/VaR ≈ 1.25 |
| 10 | VaR backtesting (Kupiec) | HW11 | expected exceptions 12.6 |
| 11 | Hazard / reduced-form credit | HW06 | P(τ≤5) = 3.63% |
| 12 | Merton structural model | HW07/HW09 | PD_Q 29.53%, PD_P 38.88% |
| 13 | CDS pricing | HW08 | approx 180 bps, full 184.55 bps |
| 14 | CVA & counterparty mitigation | HW08/HW09 | CVA ≈ 5.21 |
| 15 | Regulatory capital / RWA | HW10 | ratio 8.77%, PASS |

In [ ]:
# ── Shared setup ──────────────────────────────────────────────────────────────
import sys, os
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np
import pandas as pd
from scipy import stats

def check(label, got, expected, tol=0.01):
    """Assert |got - expected| / |expected| <= tol and print result."""
    rel = abs(got - expected) / (abs(expected) + 1e-12)
    ok  = rel <= tol
    print(f"{'✓' if ok else '✗'}  {label}: got {got:.6g}, expected {expected:.6g}  (err {rel:.2%})")
    assert ok, f"{label}: rel err {rel:.3%} > {tol:.1%}"

print('repo_root:', repo_root)

---
## §1 — Coverage Matrix & Risk-Measure Theory
*(HW03 Q1 — theoretical framing)*

**Question (HW03 Q1)**  
Define VaR and ES. Which is coherent? State the four Artzner axioms and identify which one VaR violates.

**Formulas (§2–§3)**

$$\mathrm{VaR}_\alpha(L) = \inf\{\ell : P(L > \ell) \le 1-\alpha\}$$

$$\mathrm{ES}_\alpha(L) = \frac{1}{1-\alpha}\int_\alpha^1 \mathrm{VaR}_u(L)\,du$$

**Artzner coherence axioms** — $\rho$ is coherent iff for all loss random variables:
1. *Translation invariance*: $\rho(L+c) = \rho(L) - c$
2. *Positive homogeneity*: $\rho(\lambda L) = \lambda\rho(L),\ \lambda > 0$
3. *Monotonicity*: $L_1 \le L_2 \Rightarrow \rho(L_1) \le \rho(L_2)$
4. *Sub-additivity*: $\rho(L_1+L_2) \le \rho(L_1) + \rho(L_2)$

VaR fails **axiom 4** for non-elliptic distributions; ES satisfies all four.

In [ ]:
# Numerical illustration: binary bond losses, VaR sub-additivity failure
rng   = np.random.default_rng(42)
n     = 200_000
alpha = 0.95

L1 = rng.binomial(1, 0.04, n).astype(float)   # bond 1: 4% default prob
L2 = rng.binomial(1, 0.04, n).astype(float)   # bond 2: independent

var1  = np.quantile(L1,      alpha)
var2  = np.quantile(L2,      alpha)
var12 = np.quantile(L1 + L2, alpha)

def emp_es(losses, a):
    v = np.quantile(losses, a)
    return float(np.mean(losses[losses >= v]))

es1  = emp_es(L1,      alpha)
es2  = emp_es(L2,      alpha)
es12 = emp_es(L1 + L2, alpha)

print(f'VaR₉₅(L1)           = {var1}')
print(f'VaR₉₅(L2)           = {var2}')
print(f'VaR₉₅(L1+L2)        = {var12}   (> {var1+var2}? {var12 > var1+var2}  ← sub-add. violated if True)')
print()
print(f'ES₉₅(L1)            = {es1:.4f}')
print(f'ES₉₅(L2)            = {es2:.4f}')
print(f'ES₉₅(L1)+ES₉₅(L2)  = {es1+es2:.4f}')
print(f'ES₉₅(L1+L2)        = {es12:.4f}')
assert es12 <= es1 + es2 + 0.05
print('\n✓ ES satisfies sub-additivity; VaR may not (depends on distribution)')

**Expected vs actual**  
For independent binary losses at p=4%, each marginal VaR₉₅=0 (95th pct ≈ 0), but the joint VaR₉₅ can exceed 0 under the portfolio—showing VaR is not sub-additive for these discrete distributions. ES always satisfies ES(L1+L2) ≤ ES(L1)+ES(L2).

In [ ]:
# Sub-additivity assertion
assert es12 <= es1 + es2 + 1e-6, 'ES failed sub-additivity'
print('✓ Section 1 assertions passed')

**Interpretation**  
ES averages across the entire tail above the VaR threshold, making it sensitive to severity beyond the cut-off. This tail-averaging is what forces sub-additivity and makes ES the preferred coherent risk measure in Basel IV and FRTB. VaR only marks a single quantile and can ignore severity beyond it, producing paradoxical results where "diversification" appears to increase measured risk.

---
## §2 — European Option Pricing & Delta
*(Repo case HW5_BS_DeltaFiniteDiff · `tests/test_homework_cases.py::TestHW5_BS_DeltaFiniteDiff`)*

**Question**  
ATM European call: S=85, K=85, r=4.5%, σ=30%, T=2 years.  
Compute the Black-Scholes price and delta by (a) analytic formula and (b) central finite-difference bump.

**Formulas (§5)**

$$d_1 = \frac{\ln(S/K) + (r+\frac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

$$C = S\,N(d_1) - K e^{-rT}N(d_2), \quad \Delta = N(d_1)$$

$$\Delta_{\mathrm{FD}} = \frac{C(S+h) - C(S-h)}{2h}, \quad h = 0.01S$$

In [ ]:
from src.pricing.black_scholes import bs_price, bs_delta

S, K, r, sigma, T = 85.0, 85.0, 0.045, 0.30, 2.0

price_analytic = bs_price(S, K, r, sigma, T, option_type='call')
delta_analytic = bs_delta(S, K, r, sigma, T, option_type='call')

h = 0.01 * S
delta_fd = (bs_price(S+h, K, r, sigma, T, 'call') -
            bs_price(S-h, K, r, sigma, T, 'call')) / (2*h)

print(f'Analytic price : {price_analytic:.6f}')
print(f'Analytic delta : {delta_analytic:.6f}')
print(f'FD delta (bump): {delta_fd:.6f}')

**Expected vs actual**

| Quantity | Expected | Tol |
|----------|----------|-----|
| Call price | 17.624562 | ±1% |
| Analytic Δ | 0.664313 | ±1% |
| FD Δ | ≈ analytic Δ | ±0.1% |

In [ ]:
check('BS call price',  price_analytic, 17.624562)
check('Analytic delta', delta_analytic,  0.664313)
check('FD delta',       delta_fd,        0.664313, tol=0.001)
print('\n✓ Section 2 complete')

**Interpretation**  
The ATM call at T=2 years has delta 0.664 — each option tracks about 66¢ of a $1 move in the underlying. The analytic and finite-difference deltas agree to 5 decimal places, validating the implementation. The price of ~$17.62 (≈21% of spot) reflects 2 years of 30% annual volatility accumulating in the option's time value.

---
## §3 — Delta-Hedge Intuition
*(Repo case HW3_INTEL_BSM_DELTA_HEDGE · `tests/test_homework_cases.py::TestHW3_IntelBSM_DeltaHedge`)*

**Question**  
You hold 1 200 Intel shares (S₀=24.65, K=25, r=4.7%, σ=40%, T=1.5 yr).  
How many calls must you write to make the position delta-neutral?

**Formula**

$$N_{\text{calls}} = \frac{N_{\text{shares}}}{\Delta_{\text{call}}}$$

Writing $N_{\text{calls}}$ short calls adds $-N_{\text{calls}}\cdot\Delta_{\text{call}}$ delta to offset the $+N_{\text{shares}}$ long-stock delta.

In [ ]:
from src.pricing.black_scholes import bs_price, bs_delta

S0_i, K_i, r_i, sig_i, T_i = 24.65, 25.0, 0.047, 0.40, 1.5
N_shares = 1200

call_price_i = bs_price(S0_i, K_i, r_i, sig_i, T_i, 'call')
call_delta_i = bs_delta(S0_i, K_i, r_i, sig_i, T_i, 'call')
N_calls      = N_shares / call_delta_i

print(f'Intel call price : {call_price_i:.5f}')
print(f'Intel call delta : {call_delta_i:.6f}')
print(f'Shares held      : {N_shares}')
print(f'Calls to write   : {N_calls:.1f}  → round to {round(N_calls)}')

**Expected vs actual**

| Quantity | Expected | Tol |
|----------|----------|-----|
| Call price | 5.34508 | ±1% |
| Call Δ | 0.640605 | ±1% |
| N_calls | ≈ 1 873 | ±1% |

In [ ]:
check('Intel call price', call_price_i, 5.34508)
check('Intel call delta', call_delta_i, 0.640605)
check('N_calls',          N_calls,      1873.0,  tol=0.01)
print('\n✓ Section 3 complete')

**Interpretation**  
Because each call has delta ≈ 0.641, writing ≈1 873 calls offsets the positive delta of 1 200 long shares. The hedge is only instantaneously delta-neutral: as S moves, delta drifts (gamma exposure) and the book must be dynamically rebalanced. This is the core insight behind Black-Scholes: continuous delta-hedging replicates the option payoff and justifies risk-neutral pricing.

---
## §4 — Historical Scenario VaR & ES
*(HW03 scenario: Apple + IBM · `tests/test_homework_cases.py::TestHW3_ScenarioVaR_ES`)*

**Question**  
Portfolio: 100 Apple @ 228.15, 120 IBM @ 205.23.  
Given 10 historical 1-day return scenarios, compute VaR₉₀ and ES₈₀.

**Formulas (§3)**

1. P&L per scenario: $\pi_s = \sum_i n_i S_{i,0}\, r_{i,s}$  
2. $\mathrm{VaR}_{\alpha} = $ negative of the $(1{-}\alpha)$ quantile of P&L  
3. $\mathrm{ES}_{\alpha} = $ mean of losses that equal or exceed $\mathrm{VaR}_{\alpha}$

In [ ]:
scenarios = pd.DataFrame({
    'Apple': [-0.0105, 0.0183, -0.0062,  0.0042,  0.0098,
              -0.0301, 0.0071, -0.0173,  0.0014, -0.0218],
    'IBM':   [ 0.0073,-0.0094,  0.0116, -0.0051,  0.0022,
              -0.0198, 0.0045, -0.0138,  0.0061, -0.0157],
})
init_px = {'Apple': 228.15, 'IBM': 205.23}
n_shs   = {'Apple': 100,    'IBM': 120}

pnl = sum(n_shs[c] * init_px[c] * scenarios[c] for c in scenarios.columns)
losses = -pnl.values

var_90 = np.quantile(losses, 0.90)
thr_80 = np.quantile(losses, 0.80)
es_80  = np.mean(losses[losses >= thr_80])

print('Scenario P&L (sorted):', np.sort(pnl.values))
print(f'\nVaR₉₀ = {var_90:.1f}')
print(f'ES₈₀  = {es_80:.1f}')

**Expected vs actual**

| Measure | Expected | Tol |
|---------|----------|-----|
| VaR₉₀ | 3 931.2 | ±2% |
| ES₈₀ | 3 428.6 | ±2% |

In [ ]:
check('VaR₉₀', var_90, 3931.2, tol=0.02)
check('ES₈₀',  es_80,  3428.6, tol=0.02)
print('\n✓ Section 4 complete')

**Interpretation**  
VaR₉₀≈$3 931 marks the 90th-percentile loss; there is a 10% chance of exceeding it in one day. ES₈₀≈$3 429 is the average loss conditional on being in the worst 20% of scenarios. ES₈₀ < VaR₉₀ here because averaging over a wider tail region (worst 20%) includes moderate losses that dilute the extreme values captured by VaR₉₀. With only 10 draws, both estimates have high sampling uncertainty.

---
## §5 — Single-Stock GBM VaR (Lognormal)
*(HW04 Q1 · `tests/test_homework_cases.py::TestHW4_SingleStockParamVaR`)*

**Question**  
Stock: μ=0.015%/day, σ=3.5%/day, S₀=82, n=1 400 shares.  
Compute the 5-day 99% lognormal VaR for a **long** position.

**Formula (§4 — lognormal long VaR)**

$$m_h = (\mu - \tfrac12\sigma^2)h, \quad s_h = \sigma\sqrt{h}$$

$$\mathrm{VaR}^\mathrm{long}_\alpha = V_0\bigl[1 - e^{m_h + s_h\,z_{1-\alpha}}\bigr]$$

In [ ]:
from src.risk.lognormal import var_long_lognormal

mu_d, sig_d, S0_5, n5, h5, alpha5 = 0.00015, 0.035, 82.0, 1400, 5.0, 0.99
V0_5  = n5 * S0_5
var5  = var_long_lognormal(V0_5, mu_d, sig_d, h5, alpha5)

print(f'V₀ = ${V0_5:,.0f}')
print(f'5-day 99% GBM VaR = ${var5:,.2f}')

**Expected vs actual**

| Quantity | Expected | Tol |
|----------|----------|-----|
| 5-day 99% VaR | ≈ 19 037 | ±1% |

In [ ]:
check('5d-99% lognormal VaR', var5, 19037.0, tol=0.01)
print('\n✓ Section 5 complete')

**Interpretation**  
$19 037 represents ≈16.6% of the $114 800 portfolio — substantial tail risk over a 5-day window at 99% confidence. The lognormal model correctly reflects that stock prices cannot go below zero: the loss distribution has a fat right tail (large losses are possible) but is bounded on the left. This gives a slightly smaller VaR than the delta-normal approximation, which ignores the log-convexity.

---
## §6 — Two-Stock Parametric VaR
*(HW04 Q2 · `tests/test_homework_cases.py::TestHW4_TwoStockNormalVaR`)*

**Question**  
Stock 1: n=400, S=102, μ=3.5%/yr, σ=33%/yr.  
Stock 2: n=600, S=81,  μ=2.3%/yr, σ=22%/yr. ρ=0.31. T=10/252 yr. α=99%.  
Compute the delta-normal portfolio VaR.

**Formulas (§3)**

$$\sigma_P^2 = \mathbf{w}^\top\Sigma\mathbf{w}, \quad \mathrm{VaR}_\alpha = V_0(z_\alpha\sigma_P\sqrt{T} - \mu_P T)$$

In [ ]:
n1, S1, mu1, sig1 = 400, 102.0, 0.035, 0.33
n2, S2, mu2, sig2 = 600,  81.0, 0.023, 0.22
rho6, T6, a6 = 0.31, 10/252, 0.99

V1, V2 = n1*S1, n2*S2
V0_6 = V1 + V2
w = np.array([V1/V0_6, V2/V0_6])
cov6 = np.array([[sig1**2, rho6*sig1*sig2],
                  [rho6*sig1*sig2, sig2**2]])
mu_P  = w @ np.array([mu1, mu2])
sig_P = np.sqrt(w @ cov6 @ w)
z6    = stats.norm.ppf(a6)

E_V6   = V0_6 * (1 + mu_P * T6)
std6   = V0_6 * sig_P * np.sqrt(T6)
VaR6   = V0_6 * (z6 * sig_P * np.sqrt(T6) - mu_P * T6)

print(f'V₀     = ${V0_6:,.2f}')
print(f'E[V_T] = ${E_V6:,.2f}')
print(f'Std    = ${std6:,.2f}')
print(f'VaR₉₉  = ${VaR6:,.2f}')

**Expected vs actual**

| Quantity | Expected | Tol |
|----------|----------|-----|
| V₀ | 89 400.00 | exact |
| E[V_T] | 89 501.08 | ±0.1% |
| Std dev | 3 915.34 | ±1% |
| VaR₉₉ | 9 007.37 | ±1% |

In [ ]:
check('V₀',     V0_6,  89400.00,  tol=1e-9)
check('E[V_T]', E_V6,  89501.08,  tol=0.001)
check('Std dev', std6,  3915.34,  tol=0.01)
check('VaR₉₉',  VaR6,  9007.37,  tol=0.01)
print('\n✓ Section 6 complete')

**Interpretation**  
Correlation ρ=0.31 provides moderate diversification: perfect correlation would give VaR≈$9 500; independence would give ≈$8 100. At ρ=0.31 the answer falls between these. Parametric VaR is fast but assumes normality — real-world return distributions have fatter tails, which would push actual VaR somewhat higher.

---
## §7 — Rolling Window vs EWMA Calibration
*(HW05 · `tests/test_homework_cases.py::TestHW5_Lambda20PctHeuristic`)*

**Question**  
Derive the EWMA decay factor λ such that a window of N trading days retains the oldest weight at 20% of the newest.

**Formula (§6)**

EWMA variance: $\sigma_t^2 = (1-\lambda)\sum_{k=0}^\infty \lambda^k r_{t-k}^2$

20% heuristic: $\lambda^N = 0.20 \Rightarrow \lambda = 0.20^{1/N}$, where $N$ = window in trading days.

In [ ]:
TD = 252
def lam_heuristic(years, frac=0.20):
    return frac ** (1.0 / (years * TD))

lam_2y  = lam_heuristic(2)
lam_5y  = lam_heuristic(5)
lam_10y = lam_heuristic(10)

print(f'λ for 2-year  window: {lam_2y:.4f}')
print(f'λ for 5-year  window: {lam_5y:.4f}')
print(f'λ for 10-year window: {lam_10y:.4f}')

rm_yrs = np.log(0.20) / np.log(0.94) / TD
print(f'\nRiskMetrics λ=0.94 → effective window: {rm_yrs:.2f} years')

**Expected vs actual**

| λ | Expected | Tol |
|---|----------|-----|
| 2-year | 0.9968 | ±0.01% |
| 5-year | 0.9987 | ±0.01% |
| 10-year | 0.9994 | ±0.01% |

In [ ]:
check('λ(2y)',  lam_2y,  0.9968, tol=0.0001)
check('λ(5y)',  lam_5y,  0.9987, tol=0.0001)
check('λ(10y)', lam_10y, 0.9994, tol=0.0001)
print('\n✓ Section 7 complete')

**Interpretation**  
EWMA decay parameters sit very close to 1, meaning old observations retain substantial influence — a 2-year window still requires λ=0.9968 to make a 504-day-old return only 20% as important as yesterday's. EWMA reacts faster to volatility clusters than fixed rolling windows, but is also more sensitive to single large shocks. The industry-standard RiskMetrics λ=0.94 corresponds to just about 1 year of effective memory.

---
## §8 — Historical AAPL/CAT VaR & ES on Real Data
*(Bloomberg CSVs · `src/risk/historical.py`)*

**Question**  
Load Bloomberg daily closes for AAPL and CAT. Compute 5-day 95% historical VaR and ES for $10 000 positions in each.

**Method (§3 — historical simulation)**

1. Log-returns $r_t = \ln(P_t/P_{t-1})$  
2. 5-day portfolio return: sum of 5 consecutive log-returns  
3. VaR₉₅ = 95th percentile of losses; ES₉₅ = mean above VaR₉₅

In [ ]:
def load_bbg(path):
    df = pd.read_csv(path, parse_dates=['Dates'])
    df = df.rename(columns={'Dates': 'Date', 'PX_LAST': 'Close'})[['Date','Close']].dropna()
    df = df[~df['Date'].duplicated(keep='last')].sort_values('Date').set_index('Date')
    return df

data_dir  = os.path.join(repo_root, 'data')
aapl_df   = load_bbg(os.path.join(data_dir, 'AAPL_Bloomberg.csv'))
cat_df    = load_bbg(os.path.join(data_dir, 'CAT_Bloomberg.csv'))

aapl_lr = np.log(aapl_df['Close'] / aapl_df['Close'].shift(1)).dropna().iloc[-252:]
cat_lr  = np.log(cat_df['Close']  / cat_df['Close'].shift(1)).dropna().iloc[-252:]

print(f'AAPL: {len(aapl_lr)} days,  last: {aapl_lr.index[-1].date()}')
print(f'CAT:  {len(cat_lr)} days,  last: {cat_lr.index[-1].date()}')

def hist_var_es_5d(lr, V0, a=0.95):
    r5  = [lr.iloc[i:i+5].sum() for i in range(len(lr)-4)]
    pnl = V0 * (np.exp(r5) - 1)
    L   = -np.array(pnl)
    v   = np.quantile(L, a)
    e   = np.mean(L[L >= v])
    return v, e

V_aapl = V_cat = 10_000.0
var_a, es_a = hist_var_es_5d(aapl_lr, V_aapl)
var_c, es_c = hist_var_es_5d(cat_lr,  V_cat)

aligned = pd.concat([aapl_lr, cat_lr], axis=1, join='inner')
aligned.columns = ['AAPL', 'CAT']
port_lr = aligned.mean(axis=1)
var_p, es_p = hist_var_es_5d(port_lr, V_aapl + V_cat)

print(f'\nAAPL  5d-95% VaR: ${var_a:,.2f}   ES: ${es_a:,.2f}')
print(f'CAT   5d-95% VaR: ${var_c:,.2f}   ES: ${es_c:,.2f}')
print(f'Port  5d-95% VaR: ${var_p:,.2f}   ES: ${es_p:,.2f}')

**Expected vs actual** (from `tests/test_course_validation.py`)

| Measure | Reference | Note |
|---------|-----------|------|
| AAPL VaR₉₅ | ≈ 905 | varies with data vintage |
| CAT VaR₉₅ | ≈ 969 | varies with data vintage |
| Portfolio VaR₉₅ | ≈ 899 | diversification visible |

In [ ]:
assert var_a > 0 and es_a >= var_a
assert var_c > 0 and es_c >= var_c
assert var_p < var_a + var_c, 'Portfolio VaR should be less than sum (diversification)'
print(f'✓  ES ≥ VaR:   AAPL ({es_a:.0f} ≥ {var_a:.0f})   CAT ({es_c:.0f} ≥ {var_c:.0f})')
print(f'✓  Diversification: port VaR {var_p:.0f} < sum {var_a+var_c:.0f}')
print('\n✓ Section 8 complete')

**Interpretation**  
Historical simulation is non-parametric: it makes no distributional assumption and automatically captures fat tails and skewness embedded in actual return data. Diversification is clearly visible — the portfolio VaR is lower than the sum of individual VaRs because AAPL and CAT are imperfectly correlated. The key limitation: rare events not in the sample period are invisible.

---
## §9 — Monte Carlo VaR & ES
*(MC engine · `src/risk/monte_carlo.py`)*

**Question**  
Using estimated μ and Σ from §8, run 50 000 Monte Carlo paths over a 5-day horizon for the AAPL/CAT portfolio. Compute 95% VaR and ES.

**Method (§3 — MC)**

Draw $\mathbf{r} \sim \mathcal{N}(\hat{\mu}h, \hat{\Sigma}h)$ for $N$ paths, $h=5/252$. Portfolio P&L = $\sum_i V_i(e^{r_i}-1)$.

In [ ]:
rng9    = np.random.default_rng(2025)
N_mc    = 50_000
h_mc    = 5 / 252
a9      = 0.95

mu_ann  = aligned.mean() * 252
cov_ann = aligned.cov()  * 252
mu_h    = mu_ann.values  * h_mc
cov_h   = cov_ann.values * h_mc

L_chol  = np.linalg.cholesky(cov_h)
Z       = rng9.standard_normal((N_mc, 2))
R       = mu_h + Z @ L_chol.T
V_arr   = np.array([V_aapl, V_cat])
pnl_mc  = (V_arr * (np.exp(R) - 1)).sum(axis=1)
losses9 = -pnl_mc

var_mc  = np.quantile(losses9, a9)
es_mc   = np.mean(losses9[losses9 >= var_mc])

print(f'MC 5d-95% VaR : ${var_mc:,.2f}')
print(f'MC 5d-95% ES  : ${es_mc:,.2f}')
print(f'ES/VaR ratio  : {es_mc/var_mc:.3f}')

**Expected vs actual**

| Property | Expected |
|----------|---------|
| VaR > 0 | always |
| ES ≥ VaR | always |
| ES/VaR for bivariate normal | 1.1 – 1.4 |

In [ ]:
assert var_mc > 0
assert es_mc  >= var_mc
assert 1.0 < es_mc/var_mc < 2.0
print(f'✓  VaR={var_mc:.2f} > 0,  ES={es_mc:.2f} ≥ VaR,  ratio={es_mc/var_mc:.3f}')
print('\n✓ Section 9 complete')

**Interpretation**  
Monte Carlo gives the full P&L distribution, not just two numbers. This lets us price non-linear positions (options) correctly, add fat-tail distributions (Student-t draws), and compute coherent risk measures for complex portfolios. For Gaussian returns the MC VaR converges to the parametric answer as N→∞. The ES/VaR ratio near 1.25 matches bivariate normal theory.

---
## §10 — VaR Backtesting (Kupiec Test)
*(HW11 · `src/risk/backtest.py`)*

**Question**  
Backtest a 95% VaR model over 252 daily observations. Report expected exception count and Kupiec LR test statistic.

**Kupiec LR statistic (§12)**

$$LR_{\mathrm{uc}} = -2\ln\!\left[\frac{(1-\alpha)^x\,\alpha^{n-x}}{\hat{p}^x(1-\hat{p})^{n-x}}\right] \sim \chi^2_1$$

Reject $H_0$ (correct model) if $LR_{\mathrm{uc}} > 3.84$.

In [ ]:
from src.risk.backtest import kupiec_lr_test

alpha_bt = 0.95
n_bt     = 252
exp_exc  = n_bt * (1 - alpha_bt)
print(f'Expected exceptions (252 × 5%): {exp_exc}')

# Simulate a consistent exception count
rng10  = np.random.default_rng(42)
x_obs  = int(rng10.poisson(exp_exc))
print(f'Simulated exceptions            : {x_obs}')

lr_stat, p_val = kupiec_lr_test(n_bt, x_obs, 1 - alpha_bt)
print(f'\nKupiec LR stat : {lr_stat:.4f}  (χ²₁ critical @ 5% = 3.84)')
print(f'p-value        : {p_val:.4f}')
print('Result         :', 'PASS (fail to reject H₀)' if lr_stat < 3.84 else 'FAIL (reject H₀)')

**Expected vs actual**

| Quantity | Expected |
|----------|---------|
| Expected exceptions | 12.6 |
| LR stat (model correct) | < 3.84 |
| p-value (model correct) | > 0.05 |

In [ ]:
check('Expected exceptions', exp_exc, 12.6, tol=0.001)
assert lr_stat >= 0
assert 0 <= p_val <= 1
print(f'✓  LR = {lr_stat:.4f}, p = {p_val:.4f}')
print('\n✓ Section 10 complete')

**Interpretation**  
A correct 95% VaR model should be exceeded exactly 5% of the time — 12.6 days out of 252. The Kupiec LR test formalises this as a likelihood-ratio test. A parametric normal model typically passes backtests in calm periods but can fail during stress episodes (too many or too few exceptions). Regulators use a traffic-light system: <5 exceptions (green), 5-9 (yellow), ≥10 (red) — our expected count of 12.6 sits right in the yellow zone.

---
## §11 — Hazard Rate / Reduced-Form Credit Model
*(HW06 · `tests/test_course_validation.py::TestHZ01_ConstantHazard`, `TestHZ02_PiecewiseHazard`)*

**Question**  
(A) Constant λ=0.0074: find S(5), P(τ≤5), P(3<τ≤4).  
(B) Piecewise λ: [0→1: 1%, 1→2: 1.1%, 2+: 1.2%], LGD=70%, r=5%: credit spread term structure.

**Formulas (§8)**

$$S(t) = e^{-\lambda t}, \quad P(\tau \le t) = 1 - S(t), \quad P(t_1 < \tau \le t_2) = S(t_1) - S(t_2)$$

Credit spread: $s(T) \approx \lambda\cdot\mathrm{LGD}$ (flat hazard)

In [ ]:
from src.credit.hazard import (
    survival, interval_default_prob,
    survival_piecewise, credit_spread
)

lam_A = 0.0074
S5    = survival(5.0, lam_A)
PD5   = 1 - S5
P34   = interval_default_prob(3.0, 4.0, lam_A)

print('=== Part A: Constant λ = 0.0074 ===')
print(f'S(5)      = {S5:.6f}')
print(f'P(τ≤5)   = {PD5:.6f}  ({PD5*100:.4f}%)')
print(f'P(3<τ≤4) = {P34:.6f}  ({P34*100:.4f}%)')

grid11 = [0, 1, 2, 50]
haz11  = [0.01, 0.011, 0.012]
LGD11, r11 = 0.70, 0.05

print('\n=== Part B: Piecewise Hazard — Credit Spread Term Structure ===')
for T11 in [0.5, 1, 2, 3, 5, 7, 10]:
    sT = survival_piecewise(T11, grid11, haz11)
    sp = credit_spread(T11, LGD11, sT) * 10_000
    print(f'  T={T11:4.1f}y: S={sT:.6f}  spread={sp:.2f} bps')

**Expected vs actual**

| Quantity | Expected | Tol |
|----------|----------|-----|
| S(5) | 0.963700 | ±0.01% |
| P(τ≤5) | 0.036324 | ±0.01% |
| P(3<τ≤4) | 0.007211 | ±0.01% |
| Spread T=0.5 | 69.95 bps | ±1% |
| Spread T=10 | 80.44 bps | ±1% |

In [ ]:
check('S(5)',       S5,   0.963700, tol=0.0001)
check('P(τ≤5)',   PD5,  0.036324, tol=0.0001)
check('P(3<τ≤4)', P34,  0.007211, tol=0.0001)

sp05 = credit_spread(0.5, LGD11, survival_piecewise(0.5,  grid11, haz11)) * 10_000
sp10 = credit_spread(10,  LGD11, survival_piecewise(10.0, grid11, haz11)) * 10_000
check('Spread T=0.5 (bps)', sp05, 69.95, tol=0.01)
check('Spread T=10  (bps)', sp10, 80.44, tol=0.01)

print('\n✓ Section 11 complete')

**Interpretation**  
With constant hazard λ=0.74%, the 5-year survival is 96.37% — a relatively safe obligor. The piecewise case shows the credit spread rising from 69.95 to 80.44 bps as the hazard rate itself increases over time (1.0%→1.1%→1.2%), producing an upward-sloping credit spread curve. In practice, hazard rates are calibrated by bootstrapping from observed CDS quotes at liquid tenors.

---
## §12 — Merton Structural Credit Model
*(HW07 + HW09 · `tests/test_course_validation.py::TestMR01_HomeworkVII_QvsP`, `TestMR02_TargetSurvivalInversion`)*

**Question**  
(A) Firm: V₀=1.1M, B=850k, σ=28%, T=5yr, r=5.5%, μ=2.3%. Compute Q-measure PD and P-measure PD.  
(B) Find barrier B* such that Q-survival = 0.96368 (V₀=15M, σ=30%, r=5%, T=5yr).

**Formulas (§9)**

$$d_2^\nu = \frac{\ln(V_0/B) + (\nu - \frac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}, \quad \mathrm{PD}^\nu = N(-d_2^\nu)$$

Use $\nu = r$ for Q-measure, $\nu = \mu$ for P-measure.

In [ ]:
from src.credit.merton import merton_pd, merton_d1_d2, merton_implied_B

V0_A, B_A, sig_A, T_A = 1_100_000, 850_000, 0.28, 5.0
r_A, mu_A = 0.055, 0.023

PD_Q = merton_pd(V0_A, B_A, r_A,  sig_A, T_A)
PD_P = merton_pd(V0_A, B_A, mu_A, sig_A, T_A)
_, d2_Q = merton_d1_d2(V0_A, B_A, r_A,  sig_A, T_A)
_, d2_P = merton_d1_d2(V0_A, B_A, mu_A, sig_A, T_A)

print('=== Part A: Q vs P Default Probability ===')
print(f'd₂^Q = {d2_Q:.4f}   PD_Q = {PD_Q*100:.2f}%')
print(f'd₂^P = {d2_P:.4f}   PD_P = {PD_P*100:.2f}%')
print(f'μ ({mu_A}) < r ({r_A}) → PD_P > PD_Q: {mu_A < r_A}')

# Part B: inversion
V0_B, sig_B, r_B, T_B, targ = 15_000_000, 0.30, 0.05, 5.0, 0.96368
B_star = merton_implied_B(V0_B, targ, r_B, sig_B, T_B)
PD_vfy = merton_pd(V0_B, B_star, r_B, sig_B, T_B)
print(f'\n=== Part B: Implied Barrier ===')
print(f'B* = ${B_star:,.2f}')
print(f'Verify PD = {PD_vfy*100:.4f}%  (target = {(1-targ)*100:.4f}%)')

**Expected vs actual**

| Quantity | Expected | Tol |
|----------|----------|-----|
| PD_Q | 29.53% | ±1% |
| PD_P | 38.88% | ±1% |
| B* | 4 612 960.81 | ±1% |

In [ ]:
check('PD_Q', PD_Q,   0.2953, tol=0.01)
check('PD_P', PD_P,   0.3888, tol=0.01)
check('B*',   B_star, 4_612_960.81, tol=0.01)
assert PD_P > PD_Q, 'When μ < r, P-measure PD > Q-measure PD'
print('\n✓ Section 12 complete')

**Interpretation**  
Q-measure uses the risk-free rate as drift; P-measure uses the true expected return μ. Since μ=2.3% < r=5.5%, the firm grows more slowly under the real-world measure, leading to a higher default probability (38.88% vs 29.53%). This Q/P gap reflects the equity risk premium embedded in the stock's expected return. The implied barrier B*≈4.6M for a $15M firm represents the debt level consistent with the observed 3.63% default probability — useful for calibrating the model to CDS market quotes.

---
## §13 — CDS Pricing
*(HW08 · `tests/test_course_validation.py::TestCDS01_FlatApprox`, `TestCDS02_FullAnnualPaymentParSpread`)*

**Question**  
(A) Constant λ=3%, R=40%: approximate CDS par spread.  
(B) Full discrete formula: r=5%, λ=3%, R=40%, annual payments, T=5 and 10 years.

**Formulas (§10)**

Approximate: $s \approx (1-R)\lambda$

Full par spread:
$$s = \frac{(1-R)\sum_i \Delta t_i\,\bar{q}_i\,B_i}{\sum_i \Delta t_i\,S(t_i)\,B_i}$$

In [ ]:
from src.credit.cds import cds_par_spread_constant_hazard, cds_par_spread

lam13, R13, r13 = 0.03, 0.40, 0.05

sp_approx = cds_par_spread_constant_hazard(lam13, R13)
print(f'Approx CDS spread = {sp_approx*10000:.1f} bps  [(1-R)λ = {(1-R13)*lam13:.4f}]')

for T13 in [5, 10]:
    times13 = list(range(1, T13+1))
    haz13   = [lam13] * T13
    sp_full = cds_par_spread(times13, haz13, r13, R13, accrual=True)
    print(f'Full par spread T={T13}y = {sp_full*10000:.2f} bps')

**Expected vs actual**

| Quantity | Expected | Tol |
|----------|----------|-----|
| Approx spread | 180.0 bps | exact |
| Full T=5 | 184.55 bps | ±1% |
| Full T=10 | 184.55 bps | ±1% |

In [ ]:
check('Approx (bps)',  sp_approx*10000,  180.0,  tol=1e-6)

sp5  = cds_par_spread(list(range(1,6)),  [lam13]*5,  r13, R13, accrual=True)*10000
sp10 = cds_par_spread(list(range(1,11)), [lam13]*10, r13, R13, accrual=True)*10000
check('Full T=5 (bps)',  sp5,  184.55, tol=0.01)
check('Full T=10 (bps)', sp10, 184.55, tol=0.01)
print('\n✓ Section 13 complete')

**Interpretation**  
The approximate spread (1−R)λ = 180 bps is a quick mental-math anchor. The full formula gives 184.55 bps — slightly higher because it accounts for accrued premium on default (protection buyer owes a stub payment if default happens between coupon dates). The 4.55 bps difference is the accrued-premium correction. The flat CDS term structure (T=5 and T=10 both give 184.55 bps) reflects the constant hazard assumption.

---
## §14 — CVA & Counterparty Risk Mitigation
*(HW08/HW09 · `tests/test_homework_cases.py::TestHW9_DiscreteCVA` · `src/credit/cva.py`, `src/credit/mitigation.py`)*

**Question**  
Binomial tree: S₀=100, S_T={165, 45}, bond B_T=100, r=0, R=30%, PD_Q=25%.  
Price a European call (K=100) with counterparty credit risk. Compute CVA.

**Formulas (§11)**

$$p = \frac{B_T/B_0 - S_T^{\text{dn}}/S_0}{S_T^{\text{up}}/S_0 - S_T^{\text{dn}}/S_0}$$

$$\mathrm{CVA} = (1-R)\sum_i \bar{E}_i\,p_i^{\mathrm{def}}$$

In [ ]:
from src.credit.cva import cva_discrete

S0_14, ST_up, ST_dn, BT = 100.0, 165.0, 45.0, 100.0
R14, PD14 = 0.30, 0.25

p_up   = (BT/100.0 - ST_dn/S0_14) / (ST_up/S0_14 - ST_dn/S0_14)
p_dn   = 1 - p_up

K14  = 100.0
C_up = max(ST_up - K14, 0)
C_dn = max(ST_dn - K14, 0)
C0   = p_up * C_up + p_dn * C_dn   # r=0 → no discounting
EPE  = C0

CVA      = cva_discrete([EPE], [PD14], R14)
C0_risky = C0 - CVA

print(f'p_up = {p_up:.4f},  p_dn = {p_dn:.4f}')
print(f'C_up = {C_up},  C_dn = {C_dn}')
print(f'Risk-free call price C₀ = {C0:.4f}')
print(f'EPE                     = {EPE:.4f}')
print(f'CVA = (1-R)·PD·EPE     = {CVA:.4f}')
print(f'Risky call price        = {C0_risky:.4f}')

In [ ]:
# Mitigation demonstration
from src.credit.mitigation import netting_benefit, collateral_reduction

net_exp  = netting_benefit(EPE, 3.0)          # $3 offsetting trade
coll_exp = collateral_reduction(net_exp, 2.0)  # $2 collateral posted
CVA_mit  = cva_discrete([coll_exp], [PD14], R14)

print(f'\nGross EPE        = {EPE:.2f}')
print(f'After netting    = {net_exp:.2f}')
print(f'After collateral = {coll_exp:.2f}')
print(f'CVA (mitigated)  = {CVA_mit:.4f}  (vs unmitigated {CVA:.4f})')

**Expected vs actual**

| Quantity | Expected | Tol |
|----------|----------|-----|
| p_up | 0.4583 | ±0.5% |
| p_dn | 0.5417 | ±0.5% |
| CVA | ≈ 5.21 | ±2% |

In [ ]:
check('p_up',   p_up,  0.4583, tol=0.005)
check('p_down', p_dn,  0.5417, tol=0.005)
check('CVA',    CVA,   5.21,   tol=0.02)
assert CVA_mit < CVA, 'Mitigation should reduce CVA'
print('\n✓ Section 14 complete')

**Interpretation**  
CVA ≈ 5.21 is the fair-value adjustment that accounts for the possibility the counterparty defaults before settlement. Netting (reducing gross exposure by offsetting trades in the same master agreement) and collateral posting both shrink EPE and thus reduce CVA materially. Central clearing via a CCP mandates daily variation margin, driving bilateral CVA toward zero — this is the major structural change in post-crisis OTC derivatives regulation (EMIR, Dodd-Frank).

---
## §15 — Regulatory Capital / RWA
*(HW10 · `tests/test_homework_cases.py::TestHW10_RWA_Capital` · `src/risk/regulatory.py`)*

**Question**  
Bank: Cash $69k (RW=0%), Municipal bond $73k (RW=45%), Commercial loan $47k (RW=100%).  
Deposits = $182k. Equity = total assets − deposits. Compute RWA, capital ratio, PASS/FAIL vs 8%.

**Formulas (§12)**

$$\mathrm{RWA} = \sum_i w_i A_i, \quad k = \frac{\text{Equity}}{\mathrm{RWA}} \ge 8\%$$

In [ ]:
from src.risk.regulatory import risk_weighted_assets, capital_ratio

assets15  = [69_000, 73_000, 47_000]
rweights  = [0.00,   0.45,   1.00]
deposits15 = 182_000

tot_assets = sum(assets15)
equity15   = tot_assets - deposits15
rwa15      = risk_weighted_assets(assets15, rweights)
ratio15, pass15 = capital_ratio(equity15, rwa15)

print(f'Total assets : ${tot_assets:>10,}')
print(f'Deposits     : ${deposits15:>10,}')
print(f'Equity       : ${equity15:>10,}')
print(f'RWA          : ${rwa15:>10,.0f}')
print(f'Capital ratio: {ratio15*100:.4f}%')
print(f'Status       : {"PASS ✓" if pass15 else "FAIL ✗"}  (minimum 8%)')

**Expected vs actual**

| Quantity | Expected | Tol |
|----------|----------|-----|
| Total assets | 189 000 | exact |
| Equity | 7 000 | exact |
| RWA | 79 850 | exact |
| Capital ratio | 8.77% | ±0.1% |
| Status | PASS | — |

In [ ]:
check('Total assets', tot_assets, 189_000,  tol=1e-9)
check('Equity',       equity15,    7_000,   tol=1e-9)
check('RWA',          rwa15,       79_850,  tol=1e-9)
check('Capital ratio (pct)', ratio15*100, 8.7702, tol=0.001)
assert pass15, f'Should PASS at {ratio15*100:.2f}%'

print('\n' + '='*60)
print('  ALL 15 SECTIONS COMPLETE ✓')
print('  demo notebook — MATH GR 5320, Spring 2026')
print('='*60)

**Interpretation**  
The bank's 8.77% capital ratio just clears the 8% Basel III minimum, leaving a thin buffer. Cash contributes zero RWA; the $47k commercial loan contributes 100% = $47k; the municipal bond's 45% weight reflects its moderate credit quality. A real bank also faces leverage ratio requirements (equity / total assets, not risk-weighted), which at 7k/189k ≈ 3.7% would likely breach the 3% leverage floor depending on jurisdiction. DFAST stress tests would apply equity/credit shocks to check adequacy under adverse macro scenarios.